# Packaging (uv, poetry), virtual envs

*0.1 Python for GenAI · run **Setup** first*

## Setup

Settings, a configured client, and two helpers. Every cell below uses them.

In [1]:
"""Shared setup for this notebook: typed settings, configured clients, logging."""

import asyncio
import json
import logging
from concurrent.futures import ThreadPoolExecutor

from dotenv import find_dotenv
from openai import AsyncOpenAI, OpenAI
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    """All configuration in one validated object, read from the environment / .env."""

    model_config = SettingsConfigDict(env_file=find_dotenv(), extra="ignore")

    openai_api_key: SecretStr
    openai_model: str = "gpt-4o-mini"
    request_timeout_seconds: float = Field(default=30, gt=0)
    max_retries: int = Field(default=2, ge=0, le=5)


settings = Settings()

client = OpenAI(
    api_key=settings.openai_api_key.get_secret_value(),
    timeout=settings.request_timeout_seconds,
    max_retries=settings.max_retries,
)


def async_client() -> AsyncOpenAI:
    """A fresh async client per event loop (async clients are bound to the loop they run in)."""
    return AsyncOpenAI(
        api_key=settings.openai_api_key.get_secret_value(),
        timeout=settings.request_timeout_seconds,
        max_retries=settings.max_retries,
    )


def run_async(coroutine):
    """Run a coroutine from a notebook (which already has an event loop). Scripts use asyncio.run()."""
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coroutine).result()


def show(title: str, value) -> None:
    """Print a labelled, formatted JSON block."""
    print(title)
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))


logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
for noisy in ["httpx", "httpx2", "httpcore", "openai"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)

print("model:", settings.openai_model, "| timeout:", settings.request_timeout_seconds, "s")

model: gpt-4o-mini | timeout: 30.0 s


### uv

> **Problem.** CI passes, staging passes, production breaks. A transitive dependency released a new version between the two deploys; `pip install` on each machine pulled whatever was newest that day. Nobody changed a line of code.

**Idea.** Declare what you want; the lockfile records exactly what was resolved, and every machine installs from it.

**Use when** new Python projects.  
**Not when** the team is on Poetry — one tool per repo.

```mermaid
flowchart LR
    P["pyproject.toml · httpx>=0.27"] -->|uv lock| L["uv.lock · httpx==0.28.1 + 23 pins"] -->|uv sync --frozen| D[identical: laptop · CI · Docker]
```

**How it works.**
1. `uv init` creates `pyproject.toml`; `uv add httpx pydantic` records the requirement there (`httpx>=0.28`) and resolves the full tree.
2. The resolution — every package and its dependencies at one exact version — is written to `uv.lock`, which is committed.
3. `uv add --dev pytest ruff` puts tooling in a separate group so production images do not carry it.
4. `uv sync --frozen` installs exactly the lock into `.venv` and fails if `pyproject.toml` and the lock disagree — the check CI needs.
5. `uv run python ...` runs inside that environment without activating anything.

| | what happens | result |
|:--|:--|:--|
| ✗ | `pip install httpx` on each machine | different versions, different bugs |
| ✓ | `uv add httpx` → `uv sync --frozen` | same 24 pinned packages everywhere |

**Production code and its real output**

In [2]:
# uv — init, add, lock, sync, run. The lockfile pins every transitive version so CI and
# production install identical trees (`uv sync --frozen`).
import subprocess
import tempfile
import tomllib
from pathlib import Path

with tempfile.TemporaryDirectory() as folder:
    project = Path(folder) / "service"
    subprocess.run(
        ["uv", "init", "--no-workspace", "--python", "3.12", str(project)],
        capture_output=True,
        check=True,
    )
    subprocess.run(["uv", "add", "httpx", "pydantic"], cwd=project, capture_output=True, check=True)
    subprocess.run(
        ["uv", "add", "--dev", "pytest", "ruff"], cwd=project, capture_output=True, check=True
    )
    pyproject = tomllib.loads((project / "pyproject.toml").read_text())
    lock = tomllib.loads((project / "uv.lock").read_text())

print("dependencies:    ", pyproject["project"]["dependencies"])
print("dev dependencies:", pyproject["dependency-groups"]["dev"])
print("pinned in uv.lock:", len(lock["package"]), "packages")
assert len(lock["package"]) > 10

dependencies:     ['httpx>=0.28.1', 'pydantic>=2.13.5']
dev dependencies: ['pytest>=9.1.1', 'ruff>=0.16.8']
pinned in uv.lock: 19 packages


**What the output shows.** A fresh project ended with two runtime dependencies, two dev dependencies, and a lockfile pinning every transitive package — the same tree any machine will install.

**In practice**
- **commit the lock** — and use `--frozen` in CI and Docker so drift fails loudly instead of installing something new.
- **upgrade on purpose** — `uv lock --upgrade` on a schedule with the test suite, reviewed like any change — never as a side effect of a deploy.
- **pin Python** — `.python-version` in the repo; a different interpreter version is a dependency change too.
- **no pip inside** — `pip install` into a uv-managed venv is invisible to the lock; the next `uv sync` removes it and someone asks why.
- **Docker layering** — copy `pyproject.toml` + `uv.lock` and sync before copying source, so dependency layers cache across code changes.

**Alternatives** — Poetry (same model, slower) · pip + pip-tools (`requirements.in` → `requirements.txt`) · conda for native scientific stacks

**Terms** — *lockfile*: every package pinned to one version, including dependencies of dependencies · *frozen*: install exactly the lock; fail if it drifted


### poetry

> **Problem.** You join a team whose repos, CI and Dockerfiles are built around Poetry. Adding uv on the side gives two lockfiles that disagree within a week.

**Idea.** Same model as uv — `pyproject.toml` + `poetry.lock` — with its own commands.

**Use when** the repo already uses it.  
**Not when** starting fresh — uv is faster and simpler.

```
uv add httpx        ◀──▶  poetry add httpx
uv.lock             ◀──▶  poetry.lock
uv sync --frozen    ◀──▶  poetry install --sync
```

**How it works.**
1. `uvx poetry new service` creates a project without installing Poetry globally (`uvx` runs a tool in a temporary environment).
2. `poetry add httpx --lock` records the requirement and writes `poetry.lock` without installing.
3. `poetry install --sync` in CI installs exactly the lock and removes anything not in it.
4. The cell strips `VIRTUAL_ENV` from the environment first: Poetry refuses to manage a project inside another tool's activated venv.

| | what happens | result |
|:--|:--|:--|
| ✓ | `poetry add httpx --lock` | poetry.lock with every version pinned |
| ✗ | run inside another tool's venv | confused — `VIRTUAL_ENV` must be unset (done in the cell) |

**Production code and its real output**

In [3]:
# poetry — the same workflow with the other mainstream tool (run through uvx, no global install).
import os
import subprocess
import tempfile
import tomllib
from pathlib import Path

env = dict(os.environ)
env.pop("VIRTUAL_ENV", None)  # let Poetry manage its own venv
with tempfile.TemporaryDirectory() as folder:
    subprocess.run(
        ["uvx", "poetry", "new", "service"], cwd=folder, capture_output=True, check=True, env=env
    )
    project = Path(folder) / "service"
    subprocess.run(
        ["uvx", "poetry", "add", "httpx", "--lock"],
        cwd=project,
        capture_output=True,
        check=True,
        env=env,
    )
    pyproject = tomllib.loads((project / "pyproject.toml").read_text())
    lock = tomllib.loads((project / "poetry.lock").read_text())

print("dependencies:", pyproject["project"]["dependencies"])
print("pinned in poetry.lock:", len(lock["package"]), "packages")
assert "httpx" in " ".join(pyproject["project"]["dependencies"])

dependencies: ['httpx (>=0.28.1,<0.29.0)']
pinned in poetry.lock: 7 packages


**What the output shows.** Poetry produced a project with `httpx` declared and a lockfile pinning every package — the same guarantee as `uv.lock`, in its own format.

**In practice**
- **one tool** — never two managers in one repo; the lockfiles will disagree and CI will install one while developers use the other.
- **sync in CI** — `poetry install --sync --no-root` so removed packages actually leave the environment.
- **export when needed** — `poetry export -f requirements.txt` for systems that only understand pip.

**Alternatives** — uv · pip-tools

**Terms** — *uvx*: run a tool without installing it globally


### virtual envs

> **Problem.** Project A pins `openai==1.x`; project B needs `openai==2.x`. Both were installed into the system Python. Whichever was installed last wins, and the other project fails with an import error that looks like a bug in the code.

**Idea.** A private interpreter and packages per project.

**Use when** always.  
**Not when** never.

```mermaid
flowchart LR
    S[system Python — untouched] --> A[".venv A · openai 1.x"]
    S --> B[".venv B · openai 2.x"]
```

**How it works.**
1. A virtual environment is a folder (`.venv`) with its own `python` executable and its own `site-packages` for installed libraries.
2. `sys.prefix` points to that folder when the venv is active; `sys.base_prefix` points to the original installation — the cell compares them.
3. uv and Poetry create and use `.venv` automatically; `uv run` and `poetry run` execute inside it without activation.
4. The cell creates a second, empty venv and tries `import openai` there: it fails, proving the two environments share nothing.

| | what happens | result |
|:--|:--|:--|
| ✓ this kernel | `sys.prefix != sys.base_prefix` | True — inside `.venv` |
| ✓ fresh venv | `import openai` | ModuleNotFoundError — nothing installed yet |

**Production code and its real output**

In [4]:
# Virtual envs — this kernel runs in one; a fresh venv has nothing but the standard library.
import subprocess
import sys
import tempfile
from pathlib import Path

print("isolated venv:", sys.prefix != sys.base_prefix)
with tempfile.TemporaryDirectory() as folder:
    subprocess.run([sys.executable, "-m", "venv", str(Path(folder) / ".venv")], check=True)
    probe = subprocess.run(
        [str(Path(folder) / ".venv/bin/python"), "-c", "import openai"],
        capture_output=True,
        text=True,
    )
print("fresh venv, import openai ->", probe.stderr.strip().splitlines()[-1])
assert "No module named 'openai'" in probe.stderr

isolated venv: True


fresh venv, import openai -> ModuleNotFoundError: No module named 'openai'


**What the output shows.** This notebook runs inside a venv; a brand-new venv has no third-party packages at all until something installs them from a lockfile.

**In practice**
- **Docker too** — build the venv inside the image from the lockfile; never `pip install` at container start (slow, unreproducible, needs network).
- **wrong interpreter** — "module not found" for a package you just installed almost always means the editor or terminal is using a different Python — check `sys.executable`.
- **never system Python** — operating systems depend on their Python; installing into it can break the OS and needs `sudo`, which is the wrong signal.

**Alternatives** — Docker-only workflows (the container is the environment) · conda environments for native scientific stacks

**Terms** — *venv*: the folder holding a project's private Python and packages · *sys.prefix*: where the running Python keeps its packages
